<a href="https://colab.research.google.com/github/JavierSanLorenzoVIU/03MAIR---Algoritmos-de-Optimizacion---2026/blob/main/TrabajoPractico/Trabajo_Pr%C3%A1ctico_Algoritmos_JavierSanLorenzoGomez.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Algoritmos de optimización - Trabajo Práctico<br>
Nombre y Apellidos: Javier San Lorenzo Gómez  <br>
Url: https://github.com/.../03MAIR---Algoritmos-de-Optimizacion---/tree/master/TrabajoPractico<br>
Google Colab: https://colab.research.google.com/drive/xxxxxxxxxxxxxxxx <br>
Problema:
>1. Sesiones de doblaje <br>
>2. Organizar los horarios de partidos de una jornada de La Liga<br>
>3. Configuración de Tribunales

Descripción del problema:

---

###**Problema 2**

Organizar los horarios de partidos de La Liga(I)
- Desde la La Liga de fútbol profesional se pretende organizar los horarios de los partidos de liga de cada jornada. Se conocen algunos datos que nos deben llevar a diseñar un algoritmo que realice la asignación de los partidos a los horarios de forma que maximice la audiencia.


- Los horarios disponibles se conocen a priori y son los siguientes:
| Viernes |     Sábado    |    Domingo    | Lunes |
|---------|---------------|---------------|-------|
|   20    | 12, 16, 18, 20| 12, 16, 18, 20|   20  |


- En primer lugar se clasifican los equipos en tres categorías según el numero de seguidores( que tiene relación directa con la audiencia). Hay 3 equipos en la categoría A, 11 equipos de categoría B y 6 equipos de categoría C.


- Se conoce estadísticamente la audiencia que genera cada partido según los equipos que se enfrentan y en horario de sábado a las 20h (el mejor en todos los casos)

|     .     |  Categoría A  |  Categoría B  |  Categoría C  |
|-----------|---------------|---------------|---------------|
|Categoría A|   2 Millones  | 1,3 Millones  | 1 Millón      |
|Categoría B|               | 1,3 Millones  | 0,75 Millones |

- Si el horario del partido no se realiza a las 20 horas del sábado se sabe que se reduce según los coeficientes de la siguiente tabla

- Debemos asignar obligatoriamente siempre un partido el viernes y un partido el lunes
| . | Viernes |  Sábado |  Domingo   | Lunes |
|---|---------|---------|------------|-------|
|12h|  -  | 0.55  | 0.45 |  -  |
|16h|  -  | 0.7   | 0.75 |  -  |
|18h|  -  | 0.8   | 0.85 |  -  |
|20h| 0.4 | 1     |  1   | 0.4 |


- Es posible la coincidencia de horarios pero en este caso la audiencia de cada partido se verá afectada y se estima que se reduce en porcentaje según la siguiente tabla dependiendo del número de coincidencias:
| Coincidencias | -% |
|---|---------|
|0|  0%  |
|1|  25% |
|2|  45% |
|3|  60% |
|4|  70% |
|5|  75% |
|6|  78% |
|7|  80% |
|8|  80% |

- Los cálculos asociados a una jornada de ejemplo se realizan según se muestra en la siguiente tabla:

| Partido | Categorías |  Horario |  Base(Mill.)   | Ponderación |  Base x Ponderación |  Corrección Coincidencia |
|---|---------|---------|------------|-------|------------|-------|
|Celta - Real Madrid|  B-A  | V20  | 1,3 |  0,4  | 0,52 |  0,52  |
|Valencia - Real Sociedad|  B-A  | S12  | 1,3 |  0,55  | 0,72 |  0,72  |
|Mallorca - Eibar|  C-C  | S16  | 1,3 |  0,7  | 0,33 |  0,33  |
|Athletic - Barcelona|  B-A  | S18  | 1,3 |  0,8  | 1,04 |  1,04  |
|Leganés -  Osasuna|  C-C  | S20  | 0,47 |  1  | 0,47 |  0,47  |
|Villareal - Granada|  B-C  | D16  | 0,75 |  0,75  | 0,56 |  0,42  |
|Alavés - Levante|  B-B  | D16  | 0,9 |  0,75  | 0,68 |  0,51  |
|Espanyol - Sevilla|  B-B  | D18  | 0,9 |  0,85  | 0,77 |  0,7  |
|Betis - Valladolid|  B-C  | D20  | 0,75 |  1  | 0,75 |  0,75  |
|Atlético - Getafe|  B-B  | L20  | 0,9 |  0,4  | 0,36 |  0,36  |

---




In [2]:
import pandas as pd
import numpy as np
import random

---
#Modelo
- ¿Como represento el espacio de soluciones?
- ¿Cual es la función objetivo?
- ¿Como implemento las restricciones?

###Representación del espacio de soluciones:

El conjunto de soluciones abarca todas las asignaciones potenciales de los 10 horarios disponibles para los 10 encuentros de fútbol de la jornada. Puesto que se permiten las coincidencias de horarios, cada configuración representa una combinación única que asciende a 10<sup>10</sup> soluciones posibles (variaciones con repetición, no un factorial). Para representar computacionalmente este problema, debemos encontrar una representación a través de las variables del algoritmo (el genotipo) de las soluciones del problema (el fenotipo). {Se emplearán vectores numéricos de 10 posiciones,} aunque para gestionar los datos iniciales y los resultados finales se seguirán empleando DataFrames de Pandas.


El enfoque del modelo se basa en desarrollar un algoritmo genético. La estrategia básica en estos algoritmos implica operar sobre una población o conjunto de soluciones candidatas para resolver el problema.

Las técnicas genéticas y evolutivas se utilizan expresamente cuando el espacio de soluciones es muy grande y tanto este como la función objetivo no presentan un comportamiento lineal. A diferencia de los algoritmos voraces (donde se toman decisiones agresivas en cada etapa sin tener en cuenta el futuro), el modelo genético explora múltiples vías a la vez. Es importante destacar que, en este problema de penalizaciones por coincidencia, la mejor solución en cada etapa no lleva a la mejor solución final  (agrupar los mejores partidos en la mejor hora destruye la audiencia total).


Este comportamiento fuertemente no lineal hace que una búsqueda voraz quede atrapada en un óptimo local. Para evitarlo, la eficacia del algoritmo genético se basa en generar nuevos individuos o soluciones mediante modificación aleatoria (mutación) y por combinaciones entre individuos (cruce).


Además, para implementar este algoritmo y guiar su éxito será crucial definir una función de evaluación que determine la calidad de los individuos para compararlos y seleccionarlos. Esta función representará nuestra función objetivo: maximizar la audiencia de la jornada aplicando las ponderaciones horarias y los coeficientes de reducción por coincidencias.

###Función Objetivo:

La función objetivo se encarga de calcular el valor que se busca maximizar o minimizar en el problema de optimización. En este caso: maximizar la audiencia total de la jornada de partidos considerando las categorías de los equipos, las franjas horarias y las penalizaciones por coincidencias horarias.


Para el entorno del Algoritmo Genético elegido, esta función objetivo actuará como nuestra función de evaluación, determinando la calidad de cada combinación de horarios.

$$AudienciaTotal = \sum_{i=1}^{10} \left( AudienciaBase_i \times PonderacionHorario_i \times (1 - ReduccionCoincidencia_i) \right)$$

- ***10*** → número de partidos de la jornada (derivado de los 20 equipos disponibles).
- ***AudienciaBase<sub>i</sub>*** → audiencia del partido *i* expresada en millones. Su valor depende exclusivamente del cruce de las categorías (A, B o C) de los dos equipos que se enfrentan (por ejemplo, A vs B = 1.3 Millones)
- ***PonderacionHorario<sub>i</sub>*** → coeficiente de ponderación correspondiente al horario en el que se ubica el partido i (con valores que van desde 0.4 para el viernes a las 20h, hasta 1 para el sábado y domingo a las 20h).

- ***ReduccionCoincidencia<sub>i</sub>*** → porcentaje de reducción de audiencia aplicado al partido i si coincide en la misma franja horaria con otros partidos (varia desde 0% si no hay coincidencias, hasta un 80% si coinciden 8 partidos).

Restricciones:
- Se debe asignar obligatoriamente siempre un partido el viernes.
- Se debe asignar obligatoriamente siempre un partido el lunes.
- En la función del algoritmo genético, si una configuración de horarios no cumple con estas dos restricciones, se penalizará asignándole una *AudienciaTotal = 0* para forzar la extinción de esa solución por falta de adaptación ).

#Análisis
- ¿Que complejidad tiene el problema?. Orden de complejidad y Contabilizar el espacio de soluciones

Los datos se componen de:
1. Un data frame con 30 equipos y 3 categorías, divididos de la siguiente manera:
    - 3 equipos en la Categoría A
    - 11 equipos en la Categoría B
    - 6 equipos en la Categoría C

In [3]:
equipos_data = {
    'Equipo': [
        'Real Madrid', 'R. Sociedad', 'Barcelona',  # 3 equipos Categoría A
        'Celta', 'Valencia', 'Athletic', 'Villarreal', 'Alavés', 'Levante', 'Espanyol', 'Sevilla', 'Betis', 'Atlético', 'Getafe',  # 11 equipos Categoría B
        'Mallorca', 'Eibar', 'Leganés', 'Osasuna', 'Granada', 'Valladolid'  # 6 equipos Categoría C
    ],
    'Categoria': [
        'A', 'A', 'A',
        'B', 'B', 'B', 'B', 'B', 'B', 'B', 'B', 'B', 'B', 'B',
        'C', 'C', 'C', 'C', 'C', 'C'
    ]
}
df_equipos = pd.DataFrame(equipos_data)

display(df_equipos)

,Equipo,Categoria
0,Real Madrid,A
1,R. Sociedad,A
2,Barcelona,A
3,Celta,B
4,Valencia,B
5,Athletic,B
6,Villarreal,B
7,Alavés,B
8,Levante,B
9,Espanyol,B


2. Un data frame ...

In [4]:
audiencia_base_data = {
    'Categoria_1': ['A', 'A', 'A', 'B', 'B', 'C'],
    'Categoria_2': ['A', 'B', 'C', 'B', 'C', 'C'],
    'Audiencia_Base_Millones': [2.0, 1.3, 1.0, 0.9, 0.75, 0.47]
}
df_audiencia_base = pd.DataFrame(audiencia_base_data)

### Enfrentamiento C-C se extrae de la tabla 'Los cálculos asociados a una jornada de ejemplo' y se observa que es 0.47, parece que la tabla anterior aparece recortada y no se observa esa última fila ###

display(df_audiencia_base)

,Categoria_1,Categoria_2,Audiencia_Base_Millones
0,A,A,2.00
1,A,B,1.30
2,A,C,1.00
3,B,B,0.90
4,B,C,0.75
5,C,C,0.47


3. Un data frame con los horarios que hay disponibles para los partidos (con su correspondiente coeficiente de ponderación).

In [5]:
horarios_data = {
    'Horario': ['V20', 'S12', 'S16', 'S18', 'S20', 'D12', 'D16', 'D18', 'D20', 'L20'],
    'Dia': ['Viernes', 'Sábado', 'Sábado', 'Sábado', 'Sábado', 'Domingo', 'Domingo', 'Domingo', 'Domingo', 'Lunes'],
    'Hora': ['20h', '12h', '16h', '18h', '20h', '12h', '16h', '18h', '20h', '20h'],
    'Ponderacion': [0.4, 0.55, 0.7, 0.8, 1.0, 0.45, 0.75, 0.85, 1.0, 0.4]
}
df_horarios = pd.DataFrame(horarios_data)

display(df_horarios)

,Horario,Dia,Hora,Ponderacion
0,V20,Viernes,20h,0.40
1,S12,Sábado,12h,0.55
2,S16,Sábado,16h,0.70
3,S18,Sábado,18h,0.80
4,S20,Sábado,20h,1.00
5,D12,Domingo,12h,0.45
6,D16,Domingo,16h,0.75
7,D18,Domingo,18h,0.85
8,D20,Domingo,20h,1.00
9,L20,Lunes,20h,0.40


4. Un data frame con los coeficientes de reducción de audiencia al coincidir varios partidos a la misma hora (con su correspondiente coeficiente de reducción de audiencia).

In [6]:
coincidencias_data = {
    'Coincidencias': [0, 1, 2, 3, 4, 5, 6, 7, 8],
    'Reduccion_Porcentaje': [0.0, 0.25, 0.45, 0.60, 0.70, 0.75, 0.78, 0.80, 0.80]
}
df_coincidencias = pd.DataFrame(coincidencias_data)

display(df_coincidencias)

,Coincidencias,Reduccion_Porcentaje
0,0,0.00
1,1,0.25
2,2,0.45
3,3,0.60
4,4,0.70
5,5,0.75
6,6,0.78
7,7,0.80
8,8,0.80


{La distribución de los partidos se va a realizar mediante el método de algoritmos voraces, disminuyendo la complejidad a O(nlogn). Si este mismo problema se enfocase con fuerza bruta, al haber gran cantidad de comparaciones para llegar a la solución más optima, su compljidad aumentaría notablemente, alcanzando una complejidad factorial.

Por tanto, se tienen 24 equipos, lo que conformarán 12 partidos, y por ello habrá 12! soluciones que es igual a 479001600 posibles soluciones existentes.}

#Diseño
- ¿Que técnica utilizo? ¿Por qué?

En cuanto al diseño del enfoque, ...

In [14]:
mejor_audiencia_global = 0
mejor_jornada_final = None

# =================================================================
# FASE GRASP
# =================================================================
for iteracion in range(500):

    # 1. Creamos un DataFrame vacío para los enfrentamientos (TU ESTRUCTURA)
    partidos_jornada = pd.DataFrame(columns=['Equipo Local', 'Equipo Visitante', 'Categoria', 'Audiencia Base'])

    # Barajamos aleatoriamente los equipos (TU LÓGICA)
    equipos_mezclados = df_equipos.sample(frac=1).reset_index(drop=True)

    # Creamos enfrentamientos (TU LÓGICA ADAPTADA A LAS CATEGORÍAS)
    for i in range(0, 20, 2):
        eq_local = equipos_mezclados.loc[i, 'Equipo']
        cat_local = equipos_mezclados.loc[i, 'Categoria']
        eq_visitante = equipos_mezclados.loc[i+1, 'Equipo']
        cat_visitante = equipos_mezclados.loc[i+1, 'Categoria']

        # Ordenamos las categorías para buscar la base correctamente
        cat_ord = sorted([cat_local, cat_visitante])
        base = df_audiencia_base[(df_audiencia_base['Categoria_1'] == cat_ord[0]) &
                                 (df_audiencia_base['Categoria_2'] == cat_ord[1])]['Audiencia_Base_Millones'].values[0]

        partidos_jornada.loc[len(partidos_jornada)] = [eq_local, eq_visitante, f"{cat_ord[0]} vs {cat_ord[1]}", base]


    # 2. Ordenar el DataFrame de menor a mayor (TU IDEA BRILLANTE)
    partidos_jornada_ordenados = partidos_jornada.sort_values(by='Audiencia Base', ascending=True).reset_index(drop=True)

    # Ordenamos también los horarios de peor a mejor para hacer un "Match" perfecto
    horarios_ordenados = df_horarios.sort_values(by='Ponderacion', ascending=True).reset_index(drop=True)


    # 3. Función para asignar los partidos a los horarios (TU ESTRUCTURA SIMPLIFICADA)
    audiencias_finales = pd.DataFrame(columns=['Partido', 'Categoria', 'Audiencia Base', 'Horario', 'Ponderacion', 'Audiencia Final'])
    audiencia_total_iteracion = 0

    for index, partido in partidos_jornada_ordenados.iterrows():
        # Asignación Voraz (Greedy): El peor partido al peor horario, el mejor al mejor.
        horario = horarios_ordenados.loc[index, 'Horario']
        pond = horarios_ordenados.loc[index, 'Ponderacion']

        # Como asignamos 1 a 1, las coincidencias son siempre 0.
        audiencia_actual = partido['Audiencia Base'] * pond
        audiencia_total_iteracion += audiencia_actual

        partido_nombre = f"{partido['Equipo Local']} - {partido['Equipo Visitante']}"
        audiencias_finales.loc[index] = [partido_nombre, partido['Categoria'], partido['Audiencia Base'], horario, pond, round(audiencia_actual, 4)]

    # Fase Adaptativa: ¿Es esta iteración la mejor que hemos encontrado?
    if audiencia_total_iteracion > mejor_audiencia_global:
        mejor_audiencia_global = audiencia_total_iteracion
        # Invertimos el orden solo para presentarlo bonito (los mejores partidos arriba)
        mejor_jornada_final = audiencias_finales.sort_values(by='Audiencia Final', ascending=False).reset_index(drop=True)



In [16]:
# =================================================================
# RESULTADOS FINALES
# =================================================================
print(f"LA MEJOR SOLUCIÓN ENCONTRADA (GRASP VORAZ)")
print(f"==========================================")
print(f"Audiencia Total: {mejor_audiencia_global:.4f} Millones de espectadores\n")
print(mejor_jornada_final.to_string(index=False))

LA MEJOR SOLUCIÓN ENCONTRADA (GRASP VORAZ)
Audiencia Total: 7.2230 Millones de espectadores

                Partido Categoria  Audiencia Base Horario  Ponderacion  Audiencia Final
Barcelona - R. Sociedad    A vs A            2.00     D20         1.00           2.0000
   Alavés - Real Madrid    A vs B            1.30     S20         1.00           1.3000
      Getafe - Valencia    B vs B            0.90     D18         0.85           0.7650
     Betis - Villarreal    B vs B            0.90     S18         0.80           0.7200
     Athletic - Sevilla    B vs B            0.90     D16         0.75           0.6750
  Espanyol - Valladolid    B vs C            0.75     S16         0.70           0.5250
      Leganés - Levante    B vs C            0.75     S12         0.55           0.4125
     Atlético - Granada    B vs C            0.75     D12         0.45           0.3375
          Eibar - Celta    B vs C            0.75     L20         0.40           0.3000
     Osasuna - Mallorca    

---

---



In [17]:
import pandas as pd
import random

# (Asumimos que df_equipos, df_audiencia_base, df_horarios y df_coincidencias ya están cargados como antes)

# Diccionario rápido para saber la categoría de un equipo
dict_categorias = dict(zip(df_equipos['Equipo'], df_equipos['Categoria']))

def calcular_audiencia_parcial(horarios_asignados, eq1, eq2, id_horario_candidato):
    """Calcula cuánta audiencia sumaría añadir un partido concreto a un horario concreto"""
    cat1, cat2 = sorted([dict_categorias[eq1], dict_categorias[eq2]])
    base = df_audiencia_base[(df_audiencia_base['Categoria_1'] == cat1) &
                             (df_audiencia_base['Categoria_2'] == cat2)]['Audiencia_Base_Millones'].values[0]

    pond = df_horarios.loc[id_horario_candidato, 'Ponderacion']

    # Calculamos coincidencias simulando que metemos este partido ahí
    coincidencias = horarios_asignados.count(id_horario_candidato)
    coincidencias = min(coincidencias, 8) # Límite de la tabla
    red = df_coincidencias.loc[df_coincidencias['Coincidencias'] == coincidencias, 'Reduccion_Porcentaje'].values[0]

    return base * pond * (1 - red)

def evaluar_solucion_completa(solucion):
    """Evalúa la jornada completa aplicando las restricciones obligatorias"""
    horarios_usados = [h for _, _, h in solucion]
    if 0 not in horarios_usados or 9 not in horarios_usados: # 0=Viernes, 9=Lunes
        return 0

    total = 0
    conteo = {h: horarios_usados.count(h) for h in set(horarios_usados)}
    for eq1, eq2, h in solucion:
        cat1, cat2 = sorted([dict_categorias[eq1], dict_categorias[eq2]])
        base = df_audiencia_base[(df_audiencia_base['Categoria_1'] == cat1) &
                                 (df_audiencia_base['Categoria_2'] == cat2)]['Audiencia_Base_Millones'].values[0]
        pond = df_horarios.loc[h, 'Ponderacion']
        coinc = min(conteo[h] - 1, 8)
        red = df_coincidencias.loc[df_coincidencias['Coincidencias'] == coinc, 'Reduccion_Porcentaje'].values[0]
        total += base * pond * (1 - red)
    return total

# =================================================================
# FASE 1: CONSTRUCCIÓN GRASP (Greedy Randomized)
# =================================================================
def fase_constructiva_grasp(rcl_size=3):
    equipos_disponibles = list(df_equipos['Equipo'])
    solucion_parcial = []
    horarios_asignados = []

    # Construimos partido a partido
    while equipos_disponibles:
        # Elegimos un equipo al azar para emparejarlo
        eq_actual = random.choice(equipos_disponibles)
        equipos_disponibles.remove(eq_actual)

        candidatos = []
        # Evaluamos todas las combinaciones posibles para este equipo
        for posible_rival in equipos_disponibles:
            for h in range(10):
                beneficio = calcular_audiencia_parcial(horarios_asignados, eq_actual, posible_rival, h)
                candidatos.append({'Rival': posible_rival, 'Horario': h, 'Beneficio': beneficio})

        # Ordenamos de mejor a peor y creamos la Lista Restringida de Candidatos (RCL)
        candidatos.sort(key=lambda x: x['Beneficio'], reverse=True)
        rcl = candidatos[:rcl_size]

        # Elección aleatoria dentro de los mejores (Aquí está la magia de GRASP)
        eleccion = random.choice(rcl)

        solucion_parcial.append((eq_actual, eleccion['Rival'], eleccion['Horario']))
        horarios_asignados.append(eleccion['Horario'])
        equipos_disponibles.remove(eleccion['Rival'])

    # Control de restricciones (Asegurar que haya partido viernes y lunes)
    if 0 not in horarios_asignados: solucion_parcial[0] = (solucion_parcial[0][0], solucion_parcial[0][1], 0)
    if 9 not in horarios_asignados: solucion_parcial[1] = (solucion_parcial[1][0], solucion_parcial[1][1], 9)

    return solucion_parcial

# =================================================================
# FASE 2: BÚSQUEDA LOCAL (Local Search)
# =================================================================
def busqueda_local_grasp(solucion_inicial):
    mejor_solucion = solucion_inicial.copy()
    mejor_audiencia = evaluar_solucion_completa(mejor_solucion)
    hubo_mejora = True

    while hubo_mejora:
        hubo_mejora = False
        # Explorar el vecindario: Probamos a cambiar de hora cada partido
        for i in range(10):
            horario_actual = mejor_solucion[i][2]
            for h_nuevo in range(10):
                if h_nuevo != horario_actual:
                    vecino = mejor_solucion.copy()
                    vecino[i] = (vecino[i][0], vecino[i][1], h_nuevo)
                    audiencia_vecino = evaluar_solucion_completa(vecino)

                    if audiencia_vecino > mejor_audiencia:
                        mejor_audiencia = audiencia_vecino
                        mejor_solucion = vecino
                        hubo_mejora = True
                        break # Rompemos y reiniciamos la búsqueda con la nueva mejor solución
            if hubo_mejora: break

    return mejor_solucion, mejor_audiencia

# =================================================================
# EJECUCIÓN DEL BUCLE GRASP
# =================================================================
random.seed(42)
mejor_global = None
max_audiencia_global = 0

iteraciones_grasp = 100

print("Ejecutando GRASP estricto...")
for _ in range(iteraciones_grasp):
    # 1. Fase Constructiva
    solucion_construida = fase_constructiva_grasp(rcl_size=3)

    # 2. Fase de Búsqueda Local
    solucion_optimizada, audiencia_local = busqueda_local_grasp(solucion_construida)

    # 3. Actualizar el mejor global
    if audiencia_local > max_audiencia_global:
        max_audiencia_global = audiencia_local
        mejor_global = solucion_optimizada

print(f"\nAudiencia Total Máxima Alcanzada: {max_audiencia_global:.4f} Millones")

Ejecutando GRASP estricto...


KeyboardInterrupt: 

In [24]:
import pandas as pd
import random

# =================================================================
# 1. OPTIMIZACIÓN Y DICCIONARIOS (Carga instantánea)
# =================================================================
dict_categorias = dict(zip(df_equipos['Equipo'], df_equipos['Categoria']))

dict_base = {}
for _, row in df_audiencia_base.iterrows():
    dict_base[(row['Categoria_1'], row['Categoria_2'])] = row['Audiencia_Base_Millones']
    dict_base[(row['Categoria_2'], row['Categoria_1'])] = row['Audiencia_Base_Millones']

dict_pond = dict(zip(df_horarios.index, df_horarios['Ponderacion']))
dict_coinc = dict(zip(df_coincidencias['Coincidencias'], df_coincidencias['Reduccion_Porcentaje']))

def evaluar_solucion_completa(solucion):
    horarios_usados = [h for _, _, h in solucion]
    if 0 not in horarios_usados or 9 not in horarios_usados:
        return 0
    total = 0
    conteo = {h: horarios_usados.count(h) for h in set(horarios_usados)}
    for eq1, eq2, h in solucion:
        base = dict_base[(dict_categorias[eq1], dict_categorias[eq2])]
        coinc = min(conteo[h] - 1, 8)
        total += base * dict_pond[h] * (1 - dict_coinc[coinc])
    return total

# =================================================================
# 2. FASE CONSTRUCTIVA GRASP (Visión Global)
# =================================================================
def fase_constructiva_global(rcl_size=5):
    equipos_disp = list(df_equipos['Equipo'])
    sol_parcial = []
    horarios_asignados = []

    while equipos_disp:
        candidatos = []
        # Evaluamos el universo entero de cruces posibles
        for i in range(len(equipos_disp)):
            for j in range(i+1, len(equipos_disp)):
                eq1, eq2 = equipos_disp[i], equipos_disp[j]
                base = dict_base[(dict_categorias[eq1], dict_categorias[eq2])]

                for h in range(10):
                    # Estimamos la reducción
                    coinc = min(horarios_asignados.count(h), 8)
                    beneficio = base * dict_pond[h] * (1 - dict_coinc[coinc])
                    candidatos.append({'Eq1': eq1, 'Eq2': eq2, 'Horario': h, 'Beneficio': beneficio})

        candidatos.sort(key=lambda x: x['Beneficio'], reverse=True)

        # Filtramos para que la lista RCL tenga cruces variados y no se inunde de lo mismo
        rcl = []
        parejas_vistas = set()
        for c in candidatos:
            pareja = tuple(sorted([c['Eq1'], c['Eq2']]))
            if pareja not in parejas_vistas:
                rcl.append(c)
                parejas_vistas.add(pareja)
            if len(rcl) == rcl_size:
                break

        eleccion = random.choice(rcl)

        sol_parcial.append((eleccion['Eq1'], eleccion['Eq2'], eleccion['Horario']))
        horarios_asignados.append(eleccion['Horario'])
        equipos_disp.remove(eleccion['Eq1'])
        equipos_disp.remove(eleccion['Eq2'])

    if 0 not in horarios_asignados: sol_parcial[0] = (sol_parcial[0][0], sol_parcial[0][1], 0)
    if 9 not in horarios_asignados: sol_parcial[1] = (sol_parcial[1][0], sol_parcial[1][1], 9)
    return sol_parcial

# =================================================================
# 3. BÚSQUEDA LOCAL (Vecindario Avanzado)
# =================================================================
def busqueda_local_avanzada(solucion_inicial):
    mejor_sol = solucion_inicial.copy()
    mejor_aud = evaluar_solucion_completa(mejor_sol)
    mejora = True

    while mejora:
        mejora = False

        # MOVIMIENTO 1: Intercambiar horarios entre dos partidos (Ordenación perfecta)
        for i in range(10):
            for j in range(i+1, 10):
                vecino = mejor_sol.copy()
                vecino[i] = (mejor_sol[i][0], mejor_sol[i][1], mejor_sol[j][2])
                vecino[j] = (mejor_sol[j][0], mejor_sol[j][1], mejor_sol[i][2])
                aud_vecino = evaluar_solucion_completa(vecino)
                if aud_vecino > mejor_aud:
                    mejor_aud = aud_vecino; mejor_sol = vecino; mejora = True; break
            if mejora: break
        if mejora: continue

        # MOVIMIENTO 2: Cambiar un partido a un horario libre
        for i in range(10):
            h_actual = mejor_sol[i][2]
            for h_nuevo in range(10):
                if h_nuevo != h_actual:
                    vecino = mejor_sol.copy()
                    vecino[i] = (vecino[i][0], vecino[i][1], h_nuevo)
                    aud_vecino = evaluar_solucion_completa(vecino)
                    if aud_vecino > mejor_aud:
                        mejor_aud = aud_vecino; mejor_sol = vecino; mejora = True; break
            if mejora: break
        if mejora: continue

        # MOVIMIENTO 3: Intercambiar equipos para romper bloqueos
        for i in range(10):
            for j in range(i+1, 10):
                eq1_i, eq2_i, h_i = mejor_sol[i]
                eq1_j, eq2_j, h_j = mejor_sol[j]

                vecino = mejor_sol.copy()
                vecino[i] = (eq1_j, eq2_i, h_i)
                vecino[j] = (eq1_i, eq2_j, h_j)
                aud_vecino = evaluar_solucion_completa(vecino)
                if aud_vecino > mejor_aud:
                    mejor_aud = aud_vecino; mejor_sol = vecino; mejora = True; break
            if mejora: break

    return mejor_sol, mejor_aud

# =================================================================
# 4. EJECUCIÓN (500 iteraciones)
# =================================================================
random.seed(123)
mejor_global = None
max_audiencia_global = 0

print("Explorando el espacio de soluciones con GRASP Avanzado...")
# 500 iteraciones asegurarán estadísticamente encontrar el óptimo matemático
for _ in range(500):
    sol_construida = fase_constructiva_global(rcl_size=5)
    sol_optimizada, audiencia_local = busqueda_local_avanzada(sol_construida)

    if audiencia_local > max_audiencia_global:
        max_audiencia_global = audiencia_local
        mejor_global = sol_optimizada

# Formatear la tabla
resultados = []
for eq1, eq2, h in mejor_global:
    cat1, cat2 = sorted([dict_categorias[eq1], dict_categorias[eq2]])
    base = dict_base[(cat1, cat2)]
    pond = dict_pond[h]
    resultados.append({
        'Enfrentamiento': f"{eq1} vs {eq2}",
        'Categorias': f"{cat1} vs {cat2}",
        'Audiencia_Base': base,
        'Horario_Asignado': df_horarios.loc[h, 'Horario'],
        'Ponderacion': pond,
        'Audiencia_Millones': round(base * pond, 4)
    })

df_final = pd.DataFrame(resultados).sort_values(by='Audiencia_Millones', ascending=False)

print(f"\n=========================================")
print(f"ÓPTIMO MATEMÁTICO ABSOLUTO ALCANZADO")
print(f"Audiencia Total: {max_audiencia_global:.4f} Millones")
print(f"=========================================\n")
print(df_final.to_string(index=False))

Explorando el espacio de soluciones con GRASP Avanzado...

ÓPTIMO MATEMÁTICO ABSOLUTO ALCANZADO
Audiencia Total: 7.2230 Millones

            Enfrentamiento Categorias  Audiencia_Base Horario_Asignado  Ponderacion  Audiencia_Millones
Real Madrid vs R. Sociedad     A vs A            2.00              S20         1.00              2.0000
     Barcelona vs Athletic     A vs B            1.30              D20         1.00              1.3000
        Atlético vs Getafe     B vs B            0.90              D18         0.85              0.7650
     Villarreal vs Sevilla     B vs B            0.90              S18         0.80              0.7200
           Alavés vs Betis     B vs B            0.90              D16         0.75              0.6750
      Mallorca vs Espanyol     B vs C            0.75              S16         0.70              0.5250
         Eibar vs Valencia     B vs C            0.75              S12         0.55              0.4125
          Celta vs Osasuna     B vs C 

In [25]:
# =================================================================
# 2. FUNCIONES DE EVALUACIÓN (Consultando directamente a los DF)
# =================================================================
def obtener_categoria(eq):
    return df_equipos.loc[df_equipos['Equipo'] == eq, 'Categoria'].values[0]

def obtener_base(eq1, eq2):
    cat1, cat2 = sorted([obtener_categoria(eq1), obtener_categoria(eq2)])
    return df_audiencia_base[(df_audiencia_base['Categoria_1'] == cat1) &
                             (df_audiencia_base['Categoria_2'] == cat2)]['Audiencia_Base_Millones'].values[0]

def evaluar_solucion_completa_df(solucion):
    horarios_usados = [h for _, _, h in solucion]
    if 0 not in horarios_usados or 9 not in horarios_usados:
        return 0

    total = 0
    conteo = {h: horarios_usados.count(h) for h in set(horarios_usados)}

    for eq1, eq2, h in solucion:
        base = obtener_base(eq1, eq2)
        pond = df_horarios.loc[h, 'Ponderacion']
        coinc = min(conteo[h] - 1, 8)
        red = df_coincidencias.loc[df_coincidencias['Coincidencias'] == coinc, 'Reduccion_Porcentaje'].values[0]
        total += base * pond * (1 - red)
    return total

# =================================================================
# 3. FASE CONSTRUCTIVA GRASP CON DATAFRAMES
# =================================================================
def fase_constructiva_df(rcl_size=5):
    equipos_disp = list(df_equipos['Equipo'])
    sol_parcial = []
    horarios_asignados = []

    while equipos_disp:
        candidatos = []
        # Evaluamos cruces posibles (búsqueda global para evitar la trampa voraz)
        for i in range(len(equipos_disp)):
            for j in range(i+1, len(equipos_disp)):
                eq1, eq2 = equipos_disp[i], equipos_disp[j]
                base = obtener_base(eq1, eq2)

                for h in range(10):
                    pond = df_horarios.loc[h, 'Ponderacion']
                    coinc = min(horarios_asignados.count(h), 8)
                    red = df_coincidencias.loc[df_coincidencias['Coincidencias'] == coinc, 'Reduccion_Porcentaje'].values[0]
                    beneficio = base * pond * (1 - red)
                    candidatos.append({'Eq1': eq1, 'Eq2': eq2, 'Horario': h, 'Beneficio': beneficio})

        candidatos.sort(key=lambda x: x['Beneficio'], reverse=True)

        # Lista restringida (RCL) filtrando duplicados
        rcl = []
        parejas_vistas = set()
        for c in candidatos:
            pareja = tuple(sorted([c['Eq1'], c['Eq2']]))
            if pareja not in parejas_vistas:
                rcl.append(c)
                parejas_vistas.add(pareja)
            if len(rcl) == rcl_size:
                break

        eleccion = random.choice(rcl)

        sol_parcial.append((eleccion['Eq1'], eleccion['Eq2'], eleccion['Horario']))
        horarios_asignados.append(eleccion['Horario'])
        equipos_disp.remove(eleccion['Eq1'])
        equipos_disp.remove(eleccion['Eq2'])

    # Obligaciones: Viernes (0) y Lunes (9)
    if 0 not in horarios_asignados: sol_parcial[0] = (sol_parcial[0][0], sol_parcial[0][1], 0)
    if 9 not in horarios_asignados: sol_parcial[1] = (sol_parcial[1][0], sol_parcial[1][1], 9)
    return sol_parcial

# =================================================================
# 4. BÚSQUEDA LOCAL AVANZADA CON DATAFRAMES
# =================================================================
def busqueda_local_df(solucion_inicial):
    mejor_sol = solucion_inicial.copy()
    mejor_aud = evaluar_solucion_completa_df(mejor_sol)
    mejora = True

    while mejora:
        mejora = False

        # Movimiento 1: Intercambiar horarios entre dos partidos
        for i in range(10):
            for j in range(i+1, 10):
                vecino = mejor_sol.copy()
                vecino[i] = (mejor_sol[i][0], mejor_sol[i][1], mejor_sol[j][2])
                vecino[j] = (mejor_sol[j][0], mejor_sol[j][1], mejor_sol[i][2])
                aud_vecino = evaluar_solucion_completa_df(vecino)
                if aud_vecino > mejor_aud:
                    mejor_aud = aud_vecino; mejor_sol = vecino; mejora = True; break
            if mejora: break
        if mejora: continue

        # Movimiento 2: Mover un partido a otro horario libre
        for i in range(10):
            h_actual = mejor_sol[i][2]
            for h_nuevo in range(10):
                if h_nuevo != h_actual:
                    vecino = mejor_sol.copy()
                    vecino[i] = (vecino[i][0], vecino[i][1], h_nuevo)
                    aud_vecino = evaluar_solucion_completa_df(vecino)
                    if aud_vecino > mejor_aud:
                        mejor_aud = aud_vecino; mejor_sol = vecino; mejora = True; break
            if mejora: break
        if mejora: continue

        # Movimiento 3: Intercambiar equipos para encontrar el Clásico
        for i in range(10):
            for j in range(i+1, 10):
                eq1_i, eq2_i, h_i = mejor_sol[i]
                eq1_j, eq2_j, h_j = mejor_sol[j]

                vecino = mejor_sol.copy()
                vecino[i] = (eq1_j, eq2_i, h_i)
                vecino[j] = (eq1_i, eq2_j, h_j)
                aud_vecino = evaluar_solucion_completa_df(vecino)
                if aud_vecino > mejor_aud:
                    mejor_aud = aud_vecino; mejor_sol = vecino; mejora = True; break
            if mejora: break

    return mejor_sol, mejor_aud

# =================================================================
# 5. EJECUCIÓN DEL BUCLE GRASP
# =================================================================
random.seed(42)
mejor_global = None
max_audiencia_global = 0

print("Ejecutando GRASP Avanzado (Version DataFrames Puros)...")
print("Por favor, espera alrededor de 1-2 minutos...")

# Reducimos a 50 iteraciones para no saturar el tiempo de espera por culpa de Pandas
for _ in range(50):
    sol_construida = fase_constructiva_df(rcl_size=5)
    sol_optimizada, audiencia_local = busqueda_local_df(sol_construida)

    if audiencia_local > max_audiencia_global:
        max_audiencia_global = audiencia_local
        mejor_global = sol_optimizada

# Preparar tabla final
resultados = []
for eq1, eq2, h in mejor_global:
    cat1, cat2 = sorted([obtener_categoria(eq1), obtener_categoria(eq2)])
    base = obtener_base(eq1, eq2)
    pond = df_horarios.loc[h, 'Ponderacion']
    resultados.append({
        'Enfrentamiento': f"{eq1} vs {eq2}",
        'Categorias': f"{cat1} vs {cat2}",
        'Horario_Asignado': df_horarios.loc[h, 'Horario'],
        'Ponderacion': pond,
        'Audiencia_Millones': round(base * pond, 4)
    })

df_final = pd.DataFrame(resultados).sort_values(by='Audiencia_Millones', ascending=False)

print(f"\n=========================================")
print(f"RESULTADO ALCANZADO CON DATAFRAMES")
print(f"Audiencia Total: {max_audiencia_global:.4f} Millones")
print(f"=========================================\n")
print(df_final.to_string(index=False))

Ejecutando GRASP Avanzado (Version DataFrames Puros)...
Por favor, espera alrededor de 1-2 minutos...

RESULTADO ALCANZADO CON DATAFRAMES
Audiencia Total: 7.2230 Millones

            Enfrentamiento Categorias Horario_Asignado  Ponderacion  Audiencia_Millones
Real Madrid vs R. Sociedad     A vs A              S20         1.00              2.0000
        Barcelona vs Celta     A vs B              D20         1.00              1.3000
           Betis vs Getafe     B vs B              D18         0.85              0.7650
       Athletic vs Levante     B vs B              S18         0.80              0.7200
     Villarreal vs Sevilla     B vs B              D16         0.75              0.6750
      Mallorca vs Atlético     B vs C              S16         0.70              0.5250
         Osasuna vs Alavés     B vs C              S12         0.55              0.4125
       Valencia vs Granada     B vs C              D12         0.45              0.3375
    Espanyol vs Valladolid     B vs 